# Day 04

Data ingestion using pandas

In [1]:
import pandas as pd 

PATH = "C:\\Users\\micro\\Documents\\ABTalksAI-Cohort\\data\\"

dfp = pd.read_csv(PATH + "plans.csv")
dfc = pd.read_csv(PATH + "claims.csv")

In [2]:
dfp.info()
dfp.head()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   plan_id            3 non-null      str  
 1   plan_name          3 non-null      str  
 2   monthly_premium    3 non-null      int64
 3   annual_deductible  3 non-null      int64
 4   copay_pct          3 non-null      int64
 5   coverage_type      3 non-null      str  
 6   network_tier       3 non-null      str  
dtypes: int64(3), str(4)
memory usage: 300.0 bytes


,plan_id,plan_name,monthly_premium,annual_deductible,copay_pct,coverage_type,network_tier
0,P101,Gold PPO,500,2000,10,PPO,Gold
1,P102,Silver HMO,300,1500,20,HMO,Silver
2,P103,Bronze HMO,150,1000,30,HMO,Bronze


In [3]:
dfc.info()
dfc.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   claim_id      5 non-null      str  
 1   member_id     5 non-null      str  
 2   plan_id       5 non-null      str  
 3   procedure     5 non-null      str  
 4   claim_amount  5 non-null      int64
 5   status        5 non-null      str  
 6   date_filed    5 non-null      str  
dtypes: int64(1), str(6)
memory usage: 412.0 bytes


,claim_id,member_id,plan_id,procedure,claim_amount,status,date_filed
0,C1001,M1001,P101,X-ray,250,Pending,2023-04-01
1,C1002,M1001,P101,Surgery,1200,Approved,2023-03-15
2,C1003,M1002,P102,X-ray,150,Denied,2023-04-05
3,C1004,M1002,P102,Surgery,900,Approved,2023-03-20
4,C1005,M1003,P103,X-ray,50,Pending,2023-04-10


In [4]:
dfc = dfc.dropna().drop_duplicates()
dfp = dfp.dropna().drop_duplicates()
dfc['date_filed'] = pd.to_datetime(dfc['date_filed'])

In [ ]:
import sqlite3

conn = sqlite3.connect(PATH + "coverage.db")
dfp.to_sql("plans", conn, if_exists="replace", index=False)
dfc.to_sql("claims", conn, if_exists="replace", index=False)
conn.commit()

cursor = conn.cursor()
cursor.execute("SELECT plan_id, monthly_premium FROM plans ORDER BY monthly_premium DESC")

rows = cursor.fetchall()
for row in rows:
    print(f"Plan: {row[0]}, Monthly Payment: {row[1]}")

    # 1) How many claims are pending for member M1001?
    cursor = conn.cursor()
    cursor.execute("""
    SELECT COUNT(*) AS pending_claims
    FROM claims
    WHERE member_id = 'M1001' AND status = 'Pending'
    """)
    pending_count = cursor.fetchone()[0]
    print(f"Pending claims for member M1001: {pending_count}")

    # 2) Which plans have a monthly premium under $400?
    cursor.execute("""
    SELECT plan_id, plan_name, monthly_premium
    FROM plans
    WHERE monthly_premium < 400
    ORDER BY monthly_premium ASC
    """)
    affordable_plans = cursor.fetchall()
    print("\nPlans with monthly premium under $400:")
    for plan in affordable_plans:
        print(f"Plan: {plan[0]} | Name: {plan[1]} | Premium: ${plan[2]}")

    # 3) JOIN between claims and plans
    cursor.execute("""
    SELECT
        c.claim_id,
        c.member_id,
        p.plan_name,
        c.procedure,
        c.status,
        c.claim_amount
    FROM claims AS c
    JOIN plans AS p
        ON c.plan_id = p.plan_id
    ORDER BY c.claim_id
    """)
    join_rows = cursor.fetchall()
    print("\nClaims joined to plan details:")
    for row in join_rows:
        print(row)

    # 4) Top-N query: most claimed procedures
    cursor.execute("""
    SELECT procedure, COUNT(*) AS claim_count
    FROM claims
    GROUP BY procedure
    ORDER BY claim_count DESC
    LIMIT 3
    """)
    top_procedures = cursor.fetchall()
    print("\nTop claimed procedures:")
    for proc in top_procedures:
        print(f"Procedure: {proc[0]} | Claims: {proc[1]}")

conn.close()

print("Created coverage.db with tables: plans, claims")

Plan: P101, Monthly Payment: 500
Plan: P102, Monthly Payment: 300
Plan: P103, Monthly Payment: 150
Created coverage.db with tables: plans, claims
